In [1]:
pip install pandas numpy scikit-learn fastapi uvicorn streamlit requests


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [11]:
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import pickle
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier

df = pd.read_csv("Online_Retail.csv", encoding="ISO-8859-1")

In [12]:
df.shape

(541909, 8)

In [13]:
df.describe()

,Quantity,UnitPrice,CustomerID
count,541909.000000,541909.000000,406829.000000
mean,9.552250,4.611114,15287.690570
std,218.081158,96.759853,1713.600303
min,-80995.000000,-11062.060000,12346.000000
25%,1.000000,1.250000,13953.000000
50%,3.000000,2.080000,15152.000000
75%,10.000000,4.130000,16791.000000
max,80995.000000,38970.000000,18287.000000


In [14]:
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/10 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/10 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/10 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/10 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/10 8:26,3.39,17850.0,United Kingdom


In [15]:
df.shape

(541909, 8)

In [16]:
df = df.dropna()
df = df[df["Quantity"] > 0]

In [17]:
df["TotalPrice"] = df["Quantity"] * df["UnitPrice"]

df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

In [18]:
snapshot = df["InvoiceDate"].max()

rfm = df.groupby("CustomerID").agg({
    "InvoiceDate": lambda x: (snapshot - x.max()).days,
    "InvoiceNo": "count",
    "TotalPrice": "sum"
})

rfm.columns = ["Recency", "Frequency", "Monetary"]


In [20]:
scaler = StandardScaler()
scaled = scaler.fit_transform(rfm)


In [23]:
kmeans = KMeans(n_clusters=4, random_state=42)
rfm["segment"] = kmeans.fit_predict(scaled)
rfm["churn"] = (rfm["Recency"] > 90).astype(int)

In [24]:
X = rfm[["Recency", "Frequency", "Monetary"]]
y = rfm["churn"]

model = RandomForestClassifier()
model.fit(X, y)

RandomForestClassifier()

In [26]:
import os
import pickle

os.makedirs("models", exist_ok=True)

pickle.dump(model, open("models/churn.pkl", "wb"))
pickle.dump(kmeans, open("models/segment.pkl", "wb"))
pickle.dump(scaler, open("models/scaler.pkl", "wb"))

print("Saved inside models folder")

Saved inside models folder
